# Step 10 — Feature Engineering

**เป้าหมาย:** สร้าง feature ใหม่จากข้อมูลที่มีอยู่ แล้วเปรียบเทียบผลกับ original features

**Feature ที่จะสร้าง:**
```
Interaction : Glucose×BMI, Glucose×Age, BMI×Age
Ratio       : Insulin/Glucose (insulin resistance proxy)
Polynomial  : Glucose², BMI²
Clinical    : Glucose_cat (normal/prediab/diab), BMI_cat (under/normal/over/obese)
Composite   : Risk_score (รวมหลายค่า)
```

## 0. Import และ Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score
import lightgbm as lgb

RANDOM_STATE = 42
FEATURE_NAMES = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
                 'Insulin','BMI','DiabetesPedigree','Age']
COLS_TO_FIX = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']

df = pd.read_csv('diabetes.csv')
X = df[FEATURE_NAMES].copy()
y = df['Outcome'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
for d in [X_train, X_test, y_train, y_test]:
    d.reset_index(drop=True, inplace=True)

# Class-Aware Imputation
fill_vals = {}
for col in COLS_TO_FIX:
    medians = []
    for cls in [0, 1]:
        mask = (X_train[col] != 0) & (y_train == cls)
        med  = X_train.loc[mask, col].median()
        medians.append(med)
        X_train.loc[(X_train[col] == 0) & (y_train == cls), col] = med
    fill_vals[col] = np.mean(medians)
for col in COLS_TO_FIX:
    X_test[col] = X_test[col].replace(0, np.nan).fillna(fill_vals[col])

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 1. สร้าง Feature ใหม่

**แนวคิดแต่ละ feature:**

| Feature | สูตร | ความหมาย |
|---------|------|----------|
| `Glucose_BMI` | Glucose × BMI | คนที่ glucose สูงและอ้วนพร้อมกัน = เสี่ยงสูงมาก |
| `Glucose_Age` | Glucose × Age | คนแก่ที่ glucose สูง = เสี่ยงมากขึ้น |
| `Insulin_Glucose` | Insulin / Glucose | ถ้าสูง = ร่างกายต้องผลิต insulin เยอะเพื่อควบคุม glucose = insulin resistance |
| `Risk_score` | รวมหลายค่า | composite score จาก domain knowledge |

In [ ]:
def add_features(df):
    d = df.copy()
    # Interaction terms
    d['Glucose_BMI']     = d['Glucose'] * d['BMI']
    d['Glucose_Age']     = d['Glucose'] * d['Age']
    d['Insulin_Glucose'] = d['Insulin'] / (d['Glucose'] + 1e-6)
    d['BMI_Age']         = d['BMI'] * d['Age']
    # Polynomial
    d['Glucose2'] = d['Glucose'] ** 2
    d['BMI2']     = d['BMI'] ** 2
    # Clinical categories
    d['Glucose_cat'] = pd.cut(d['Glucose'],
                              bins=[0, 100, 125, 999],
                              labels=[0, 1, 2]).astype(float)
    d['BMI_cat']     = pd.cut(d['BMI'],
                              bins=[0, 18.5, 25, 30, 999],
                              labels=[0, 1, 2, 3]).astype(float)
    # Composite risk score
    d['Risk_score'] = (d['Glucose']/200) + (d['BMI']/50) + (d['Age']/100) + d['DiabetesPedigree']
    return d

NEW_FEATURES  = ['Glucose_BMI','Glucose_Age','Insulin_Glucose','BMI_Age',
                  'Glucose2','BMI2','Glucose_cat','BMI_cat','Risk_score']
ALL_FEATURES  = FEATURE_NAMES + NEW_FEATURES

X_train_fe = add_features(X_train)
X_test_fe  = add_features(X_test)

print(f'Original: {len(FEATURE_NAMES)} features')
print(f'New:      {len(NEW_FEATURES)} features')
print(f'Total:    {len(ALL_FEATURES)} features')
X_train_fe[NEW_FEATURES].describe().round(2)

## 2. เปรียบเทียบโมเดล — Original vs Feature Engineering

In [ ]:
# Scale
scaler_orig = StandardScaler()
X_tr_orig = scaler_orig.fit_transform(X_train[FEATURE_NAMES])
X_te_orig = scaler_orig.transform(X_test[FEATURE_NAMES])

scaler_fe = StandardScaler()
X_tr_fe = scaler_fe.fit_transform(X_train_fe[ALL_FEATURES])
X_te_fe = scaler_fe.transform(X_test_fe[ALL_FEATURES])

scale_w = (y_train == 0).sum() / (y_train == 1).sum()
BEST = dict(n_estimators=340, learning_rate=0.019, num_leaves=10,
            min_child_samples=22, feature_fraction=0.60,
            bagging_fraction=0.69, bagging_freq=6,
            scale_pos_weight=scale_w, random_state=RANDOM_STATE, verbose=-1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = {}
for name, X_tr, X_te in [
    ('Original (8)',       X_tr_orig, X_te_orig),
    ('+ Eng (17)', X_tr_fe,  X_te_fe),
]:
    clf = lgb.LGBMClassifier(**BEST)
    clf.fit(X_tr, y_train.values)
    yp = clf.predict(X_te)
    yb = clf.predict_proba(X_te)[:, 1]
    cv_auc = cross_val_score(lgb.LGBMClassifier(**BEST), X_tr, y_train.values,
                             cv=cv, scoring='roc_auc').mean()
    results[name] = {
        'AUC (test)': round(roc_auc_score(y_test, yb), 4),
        'AUC (5-CV)': round(cv_auc, 4),
        'Recall':     round(recall_score(y_test, yp), 4),
        'Precision':  round(precision_score(y_test, yp), 4),
        'F1':         round(f1_score(y_test, yp), 4),
    }
    if '17' in name:
        fi = pd.DataFrame({'feature': ALL_FEATURES,
                           'importance': clf.feature_importances_})\
               .sort_values('importance', ascending=False)

df_res = pd.DataFrame(results).T
print(df_res.to_string())

## 3. Visualize — Bar Chart เปรียบเทียบ

In [ ]:
metrics = ['AUC (test)', 'AUC (5-CV)', 'Recall', 'Precision', 'F1']
x = np.arange(len(metrics)); width = 0.3

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#F8F8F6'); ax.set_facecolor('#F8F8F6')
for i, (name, row) in enumerate(df_res.iterrows()):
    bars = ax.bar(x + i*width, [row[m] for m in metrics],
                  width, label=name, color=['#4a90d9','#D85A30'][i], alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8.5)
ax.set_xticks(x + width/2); ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score', fontsize=10)
ax.set_title('Original Features vs Feature Engineering', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
for sp in ax.spines.values(): sp.set_linewidth(0.4)
plt.tight_layout()
plt.savefig('step10_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('บันทึก step10_comparison.png')

## 4. Feature Importance — Feature ไหนโมเดลให้ความสำคัญมากสุด

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#F8F8F6'); ax.set_facecolor('#F8F8F6')
colors = ['#D85A30' if f in NEW_FEATURES else '#4a90d9' for f in fi['feature']]
ax.barh(fi['feature'].values[::-1], fi['importance'].values[::-1],
        color=colors[::-1], alpha=0.85)
ax.legend(handles=[
    mpatches.Patch(color='#D85A30', label='New feature'),
    mpatches.Patch(color='#4a90d9', label='Original'),
], fontsize=9)
ax.set_xlabel('Feature Importance', fontsize=9)
ax.set_title('Feature Importance — Original vs New Features', fontsize=10, fontweight='bold')
for sp in ax.spines.values(): sp.set_linewidth(0.4)
plt.tight_layout()
plt.savefig('step10_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('บันทึก step10_importance.png')

## 5. สรุปและบทเรียน

### ผลที่ได้
Feature Engineering **ไม่ได้ทำให้ดีขึ้น** ใน dataset นี้ — ทำไม?

```
LightGBM ฉลาดพอที่จะค้นหา interaction เองอยู่แล้ว
Dataset เล็กเกินไป (768 rows) → feature ใหม่เพิ่ม noise มากกว่า signal
```

### แต่ Feature ใหม่บางตัวมีประโยชน์จริง
- `Glucose_BMI` และ `Insulin_Glucose` ติด Top 5 importance → โมเดล**ใช้จริง**
- ปัญหาคือ feature เดิมอย่าง `Insulin`, `SkinThickness` ยังโดดเด่นกว่า

### เมื่อไหร่ Feature Engineering ช่วยได้มาก?
```
✅ Dataset ใหญ่ (หลายหมื่นแถวขึ้นไป) — signal ชัดขึ้น
✅ โมเดลง่าย (Linear, SVM) — ต้องสร้าง interaction เอง
✅ มี domain knowledge ลึก — เช่น รู้ว่า Insulin/Glucose คือ insulin resistance index จริงๆ
❌ Dataset เล็ก + tree-based model — ต้นไม้จัดการ interaction ได้เองดีกว่า
```